# 5G RCF Anomaly Detection Demo

This notebook demonstrates **AWS Managed Prometheus RCF anomaly detection** on a live 5G network.

**Architecture**: 100 UEs → 4 gNBs → 2 AMFs → SMF → UPF (open5gs on EKS)

**Scenario**: A bad config push to AMF1 causes ~50 users to lose registration. RCF detects the anomaly.

---

## Setup

In [ ]:
import boto3
import json
import time
from datetime import datetime, timezone, timedelta
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
import requests

# Configuration
REGION = 'us-east-1'
WORKSPACE_ID = 'ws-185ff7f8-c698-4d0e-9135-945b03aeccd1'
AMP_QUERY_URL = f'https://aps-workspaces.{REGION}.amazonaws.com/workspaces/{WORKSPACE_ID}/api/v1/query'
EKS_CLUSTER = 'open5gs-amp-cluster'

session = boto3.Session(region_name=REGION)
credentials = session.get_credentials().get_frozen_credentials()

def query_amp(promql):
    """Execute a PromQL query against AMP with SigV4 auth."""
    params = {'query': promql}
    req = AWSRequest(method='POST', url=AMP_QUERY_URL, data=params)
    SigV4Auth(credentials, 'aps', REGION).add_auth(req)
    resp = requests.post(AMP_QUERY_URL, data=params, headers=dict(req.headers))
    return resp.json()['data']['result']

print(f'✓ Connected to AMP workspace: {WORKSPACE_ID}')
print(f'  Region: {REGION}')

## Step 1: Verify Baseline (Healthy State)

All 100 UEs should be registered across 2 AMFs. RCF score should be 0.

In [ ]:
# Check registered subscribers per AMF
results = query_amp('fivegs_amffunction_rm_registeredsubnbr')
print('═══ Registered Subscribers (per AMF) ═══')
total = 0
for r in results:
    pod = r['metric'].get('pod', 'unknown')
    value = int(r['value'][1])
    total += value
    print(f'  {pod}: {value} UEs')
print(f'  ─────────────────────')
print(f'  TOTAL: {total} UEs registered')
print()

# Check RCF detector state
rcf_metrics = query_amp('{__name__=~"anomaly_detector:.+", alias="5g-registered-subscribers"}')
print('═══ RCF Anomaly Detector ═══')
for r in rcf_metrics:
    name = r['metric']['__name__'].replace('anomaly_detector:', '')
    value = r['value'][1]
    print(f'  {name:12s}: {value}')
print()
print('✓ Baseline healthy' if total >= 80 else '⚠ Subscribers below expected')

## Step 2: Inject Fault (Bad Config Push to AMF1)

This simulates a human error: pushing a broken configuration to AMF1.
- AMF1 will enter CrashLoopBackOff
- ~50 UEs on AMF1 (TAC=1) will lose registration
- AMF2 (TAC=2) remains completely unaffected

In [ ]:
import subprocess

# Update kubeconfig
subprocess.run(['aws', 'eks', 'update-kubeconfig', '--region', REGION, '--name', EKS_CLUSTER], 
               capture_output=True)

# Push broken config to AMF1 (missing required time.t3512 field)
broken_config = '''
sbi:
  server:
    no_tls: true
  client:
    no_tls: true
amf:
  sbi:
    - addr: 0.0.0.0
      port: 7777
  ngap:
    - addr: 0.0.0.0
  metrics:
    - addr: 0.0.0.0
      port: 9090
  guami:
    - plmn_id: {mcc: 999, mnc: 70}
      amf_id: {region: 2, set: 1}
  tai:
    - plmn_id: {mcc: 999, mnc: 70}
      tac: 1
  plmn_support:
    - plmn_id: {mcc: 999, mnc: 70}
      s_nssai:
        - sst: 1
  security:
    integrity_order: [NIA2, NIA1, NIA0]
    ciphering_order: [NEA0, NEA1, NEA2]
  network_name:
    full: Open5GS
  amf_name: open5gs-amf1
scp:
  sbi:
    - addr: scp.open5gs.svc.cluster.local
      port: 7777
'''

print('═══ FAULT INJECTION ═══')
print('Pushing broken config to AMF1 (missing time.t3512)...')

# Create broken configmap
result = subprocess.run(
    ['kubectl', 'create', 'configmap', 'amf1-config', '-n', 'open5gs',
     f'--from-literal=amf.yaml={broken_config}',
     '--dry-run=client', '-o', 'yaml'],
    capture_output=True, text=True)
subprocess.run(['kubectl', 'apply', '-f', '-'], input=result.stdout, capture_output=True, text=True)

# Restart AMF1 to pick up broken config
subprocess.run(['kubectl', 'rollout', 'restart', 'deploy/amf1', '-n', 'open5gs'], capture_output=True)

print('✗ Bad config pushed. AMF1 will crash.')
print('  Impact: ~50 UEs will lose registration within 30s.')
print()
print('Waiting 45s for impact to be visible in metrics...')
time.sleep(45)

## Step 3: Observe the Anomaly

RCF should detect the registration drop. Let's check the metrics.

In [ ]:
# Check the impact
results = query_amp('fivegs_amffunction_rm_registeredsubnbr')
print('═══ AFTER FAULT: Registered Subscribers ═══')
total = 0
for r in results:
    pod = r['metric'].get('pod', 'unknown')
    value = int(r['value'][1])
    total += value
    status = '✗ DOWN' if 'amf1' in pod and value == 0 else '✓ OK'
    print(f'  {pod}: {value} UEs  {status}')
print(f'  ─────────────────────')
print(f'  TOTAL: {total} UEs (was 100)')
print(f'  IMPACT: {100 - total} users lost service')
print()

# Check RCF score
rcf_metrics = query_amp('{__name__=~"anomaly_detector:.+", alias="5g-registered-subscribers"}')
print('═══ RCF Anomaly Detector ═══')
for r in rcf_metrics:
    name = r['metric']['__name__'].replace('anomaly_detector:', '')
    value = r['value'][1]
    indicator = ' ← ANOMALY!' if name == 'score' and float(value) > 0 else ''
    print(f'  {name:12s}: {value}{indicator}')

## Step 4: Root Cause Analysis (Infrastructure Correlation)

The DevOps Agent would correlate the 5G anomaly with infrastructure metrics to find the root cause.

In [ ]:
print('═══ ROOT CAUSE ANALYSIS ═══')
print()

# 1. Which AMF is affected?
print('1. Per-AMF breakdown:')
results = query_amp('fivegs_amffunction_rm_registeredsubnbr')
for r in results:
    pod = r['metric'].get('pod', '')
    value = r['value'][1]
    print(f'   {pod}: {value} registered', '← AFFECTED' if int(value) == 0 else '')
print()

# 2. Pod restarts (CrashLoopBackOff indicator)
print('2. Pod restart count:')
results = query_amp('kube_pod_container_status_restarts_total{namespace="open5gs", pod=~"amf.*"}')
for r in results:
    pod = r['metric'].get('pod', '')
    restarts = r['value'][1]
    print(f'   {pod}: {restarts} restarts', '← CRASH LOOP!' if int(float(restarts)) > 2 else '')
print()

# 3. Node health
print('3. Node health:')
results = query_amp('kube_node_spec_unschedulable')
if not results:
    print('   All nodes schedulable ✓ (not a node issue)')
else:
    for r in results:
        print(f'   {r["metric"].get("node")}: unschedulable={r["value"][1]}')
print()

# Conclusion
print('═══ CONCLUSION ═══')
print('Root cause: AMF1 is in CrashLoopBackOff after a config change.')
print('Impact: ~50 users on TAC=1 lost registration.')
print('AMF2 (TAC=2) is healthy — not a cluster-wide issue.')
print('Recommendation: Rollback amf1-config ConfigMap.')

## Step 5: Recovery

Restore the valid configuration to AMF1.

In [ ]:
valid_config = '''
sbi:
  server:
    no_tls: true
  client:
    no_tls: true

time:
  t3512:
    value: 540

amf:
  sbi:
    - addr: 0.0.0.0
      port: 7777
  ngap:
    - addr: 0.0.0.0
  metrics:
    - addr: 0.0.0.0
      port: 9090
  guami:
    - plmn_id: {mcc: 999, mnc: 70}
      amf_id: {region: 2, set: 1}
  tai:
    - plmn_id: {mcc: 999, mnc: 70}
      tac: 1
  plmn_support:
    - plmn_id: {mcc: 999, mnc: 70}
      s_nssai:
        - sst: 1
  security:
    integrity_order: [NIA2, NIA1, NIA0]
    ciphering_order: [NEA0, NEA1, NEA2]
  network_name:
    full: Open5GS
  amf_name: open5gs-amf1
scp:
  sbi:
    - addr: scp.open5gs.svc.cluster.local
      port: 7777
'''

print('═══ RECOVERY ═══')
print('Restoring valid config to AMF1...')

result = subprocess.run(
    ['kubectl', 'create', 'configmap', 'amf1-config', '-n', 'open5gs',
     f'--from-literal=amf.yaml={valid_config}',
     '--dry-run=client', '-o', 'yaml'],
    capture_output=True, text=True)
subprocess.run(['kubectl', 'apply', '-f', '-'], input=result.stdout, capture_output=True, text=True)
subprocess.run(['kubectl', 'rollout', 'restart', 'deploy/amf1', '-n', 'open5gs'], capture_output=True)

print('✓ Valid config restored. Waiting 60s for UEs to re-register...')
time.sleep(60)

# Verify recovery
results = query_amp('sum(fivegs_amffunction_rm_registeredsubnbr)')
total = int(results[0]['value'][1]) if results else 0
print(f'\n═══ POST-RECOVERY ═══')
print(f'  Registered subscribers: {total}')
print(f'  Status: {"✓ RECOVERED" if total >= 80 else "⏳ Still recovering..."}')

## Summary

| Phase | registeredsubnbr | RCF Score | Root Cause |
|---|---|---|---|
| Healthy | 100 | 0 | — |
| Fault injected | ~50 | >0.1 | AMF1 CrashLoopBackOff (bad config) |
| Recovered | 100 | 0 | Config rolled back |

### Key Takeaways
1. **RCF detects onset** — the moment subscribers drop, score spikes
2. **Infra correlation** — pod restarts + per-AMF breakdown pinpoints root cause
3. **Partial impact** — only TAC=1 users affected; TAC=2 is healthy
4. **Fast recovery** — fix config, restart, UEs auto-re-register